In [3]:
import shutil
import os

print("🧹 Début du nettoyage des dossiers fantômes...")
base_dir = "/content/drive/MyDrive/UTBM_PF22/datasets/Animals-10/processed_multifidelity"

# On va vérifier les 3 emplacements
for sub in ['train/HF', 'train/BF', 'test']:
    bad_folder = os.path.join(base_dir, sub, 'raw-img')
    if os.path.exists(bad_folder):
        shutil.rmtree(bad_folder) # Supprime le dossier et son contenu
        print(f"🗑️ Dossier supprimé : {sub}/raw-img")
    else:
        print(f"✅ Aucun dossier fantôme dans : {sub}")

print("✨ Nettoyage terminé ! Tu peux relancer l'entraînement.")

🧹 Début du nettoyage des dossiers fantômes...
✅ Aucun dossier fantôme dans : train/HF
✅ Aucun dossier fantôme dans : train/BF
✅ Aucun dossier fantôme dans : test
✨ Nettoyage terminé ! Tu peux relancer l'entraînement.


In [5]:
import zipfile
import time
import os
import shutil
from google.colab import drive

print("--- 🚀 CHARGEMENT ULTRA-RAPIDE DU DATASET ---")

# 1. Montage et synchronisation forcée du Drive (LA SOLUTION EST ICI)
drive.mount('/content/drive', force_remount=True)

# Chemins
zip_source = "/content/drive/MyDrive/UTBM_PF22/datasets/Animals-10/dataset_multifidelity.zip"
local_dest = "/content/processed_multifidelity"
local_zip = "/content/dataset.zip"

# 2. Vérification que le fichier est bien détecté
if not os.path.exists(zip_source):
    print(f"❌ ERREUR : Colab ne voit toujours pas le fichier à cet emplacement :")
    print(zip_source)
    print("Vérifie l'orthographe exacte des dossiers (majuscules/minuscules).")
else:
    print("✅ Fichier ZIP détecté sur le Drive !")
    
    if not os.path.exists(local_dest):
        print("⏳ Étape 1 : Copie du ZIP vers le SSD local (quelques secondes)...")
        start = time.time()
        shutil.copy2(zip_source, local_zip) 
        
        print("⏳ Étape 2 : Extraction des images sur le SSD local...")
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall("/content/") 
            
        os.remove(local_zip) # Nettoyage
        
        if os.path.exists(local_dest):
            print(f"✅ Terminé ! 26 000 images prêtes en {(time.time()-start):.2f} secondes !")
    else:
        print("✅ Les données sont déjà décompressées et prêtes sur le SSD local.")

Mounted at /content/drive
✅ Fichier ZIP détecté sur le Drive !
⏳ Étape 1 : Copie du ZIP vers le SSD local (quelques secondes)...
⏳ Étape 2 : Extraction des images sur le SSD local...
✅ Terminé ! 26 000 images prêtes en 23.55 secondes !


In [2]:
import zipfile
import time
import os
import shutil
from google.colab import drive

print("--- 🚀 DÉMARRAGE DE LA SESSION PRO ---")

# 1. On branche le Drive
drive.mount('/content/drive', force_remount=True)

zip_source = "/content/drive/MyDrive/UTBM_PF22/datasets/Animals-10/dataset_multifidelity.zip"
local_dest = "/content/processed_multifidelity"

# 2. On prépare les données sur le SSD local
if not os.path.exists(local_dest):
    print("⏳ Copie et extraction du Dataset sur le SSD...")
    start = time.time()
    shutil.copy2(zip_source, "/content/dataset.zip") 
    
    with zipfile.ZipFile("/content/dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("/content/") 
        
    os.remove("/content/dataset.zip") 
    print(f"✅ Données prêtes en {(time.time()-start):.2f} secondes !")
else:
    print("✅ Les données sont déjà prêtes sur le SSD local.")

Mounted at /content/drive
⏳ Copie et extraction du Dataset sur le SSD...
✅ Données prêtes en 21.18 secondes !


In [4]:
import sys
import os
import importlib
import torch

print(f"🔥 Matériel : {'GPU PRO ACTIVÉ' if torch.cuda.is_available() else 'CPU (Erreur)'}")

src_path = "/content/drive/MyDrive/UTBM_PF22/src"

if not os.path.exists(src_path):
    print(f"❌ ERREUR : Le dossier {src_path} est introuvable sur le Drive.")
else:
    fichiers = os.listdir(src_path)
    if "train_baselines.py" not in fichiers:
        print(f"❌ ERREUR : 'train_baselines.py' n'est pas dans le dossier src !")
        print(f"💡 Fichiers trouvés : {fichiers}")
    else:
        # --- LA CORRECTION MAGIQUE EST ICI ---
        # 1. On force ton dossier en priorité NUMÉRO 1 absolue
        if src_path not in sys.path:
            sys.path.insert(0, src_path) 
        
        # 2. On ordonne à Python d'oublier ses anciens échecs d'importation
        importlib.invalidate_caches()

        # 3. Importation
        import train_baselines
        importlib.reload(train_baselines)
        from train_baselines import run_baseline

        try:
            print("\n🏁 DÉMARRAGE DU TEST D'INTÉGRATION (1 Époque)...")
            
            run_baseline(mode="HF", epochs=1)
            run_baseline(mode="BF", epochs=1)
            run_baseline(mode="MIXTE", epochs=1)
            
            print("\n🎉 LE TEST EST UN SUCCÈS ABSOLU ! TOUT FONCTIONNE !")
            
        except Exception as e:
            print(f"❌ ERREUR pendant l'entraînement : {e}")

🔥 Matériel : GPU PRO ACTIVÉ

🏁 DÉMARRAGE DU TEST D'INTÉGRATION (1 Époque)...

🚀 DÉMARRAGE BASELINE : HF
🔥 Entraînement en cours sur : cuda
📦 Images d'entraînement : 2352
💰 Coût par époque : 23520 CA | Coût total estimé : 23520 CA
Époque 1/1 | Loss: 2.2092
⏱️ Entraînement terminé en 0.18 minutes.

--- RÉSULTATS D'ÉVALUATION ---
📊 Précision Test HF (Propre) : 21.50%
📊 Précision Test BF (Bruité) : 13.27%
📊 Précision Mixte (Moyenne) : 17.39%
💾 Résultats et Modèle sauvegardés dans /content/drive/MyDrive/UTBM_PF22/results

🚀 DÉMARRAGE BASELINE : BF
🔥 Entraînement en cours sur : cuda
📦 Images d'entraînement : 21213
💰 Coût par époque : 21213 CA | Coût total estimé : 21213 CA
Époque 1/1 | Loss: 1.7878
⏱️ Entraînement terminé en 1.25 minutes.

--- RÉSULTATS D'ÉVALUATION ---
📊 Précision Test HF (Propre) : 43.61%
📊 Précision Test BF (Bruité) : 34.05%
📊 Précision Mixte (Moyenne) : 38.83%
💾 Résultats et Modèle sauvegardés dans /content/drive/MyDrive/UTBM_PF22/results

🚀 DÉMARRAGE BASELINE : MIXTE
🔥 